# Capítulo 6 — Conhecimento e memória

**Livro:** começa na página **189** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## Foundational Approaches to Memory · PDF p. 191

#### 🧪 Exercício: Janela deslizante + resumo

Quando a conversa passar de 6 mensagens, resuma as mais antigas com o modelo e guarde só o resumo + as 4 últimas. Simule 10 trocas e mostre o contexto final.

In [ ]:
historico, resumo = [], ""
for i in range(10):
    historico.append(f"usuário: minha {i}ª dúvida é sobre o produto {random.choice(['teclado', 'mouse', 'monitor'])}")
    if len(historico) > 6:
        antigas, historico = historico[:-4], historico[-4:]
        resumo = perguntar("Resuma em uma frase o que o usuário perguntou:\n" + resumo + "\n" + "\n".join(antigas))
print("RESUMO:", resumo); print("RECENTES:", historico)

## Semantic Memory and Vector Stores · PDF p. 194

#### 🧪 Exercício: Memória semântica de fatos do usuário

Guarde fatos sobre o usuário ("prefere entrega à tarde", "tem um gato", ...) e recupere os 2 mais relevantes para uma nova mensagem por similaridade TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
fatos = ["prefere entrega à tarde", "mora em apartamento sem elevador", "tem um gato que arranha móveis",
         "comprou uma cadeira em março", "paga sempre com Pix", "trabalha de casa usando dois monitores"]
vec = TfidfVectorizer().fit(fatos + ["entrega cadeira monitor gato"])
def lembrar(msg, k=2):
    s = (vec.transform(fatos) @ vec.transform([msg]).T).toarray().ravel()
    return [fatos[i] for i in s.argsort()[::-1][:k]]
print(lembrar("quero comprar outro monitor"), lembrar("que horário vocês entregam?"))

## GraphRAG · PDF p. 201

#### 🧪 Exercício: Grafo de conhecimento com perguntas de vários saltos

Guarde triplas (sujeito, relação, objeto) sobre produtos e fornecedores e responda, navegando o grafo, "em que país é fabricado o monitor?" (monitor → fornecedor → país).

In [ ]:
triplas = [("monitor", "fornecido_por", "VisaoTech"), ("teclado", "fornecido_por", "Teclar"), ("VisaoTech", "sediada_em", "Coreia do Sul"),
           ("Teclar", "sediada_em", "Brasil"), ("monitor", "categoria", "vídeo")]
def vizinhos(no, rel): return [o for s, r, o in triplas if s == no and r == rel]
def caminho(inicio, relacoes):
    nos = [inicio]
    for rel in relacoes: nos = [v for n in nos for v in vizinhos(n, rel)]
    return nos
print(caminho("monitor", ["fornecido_por", "sediada_em"]))